## Annotation Substantia nigra

Add neuronal types and immune cells
`docker pull alexanrna/scanpy-r:1.10-v6`

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import numpy as np 
from scipy import stats
import glob 
import os
from matplotlib import pyplot as plt
import datetime
import anndata as ad
from scipy import sparse
from anndata import AnnData

In [ ]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()

sc.settings.set_figure_params(
    dpi=80,
    facecolor="white",
    frameon=False,
)


In [ ]:
path = '/hdf5/20250220_all_data_integrated_filtered_umap_scANVI_metadata.sn.h5ad'
adata = sc.read_h5ad(path)

In [ ]:
adata

In [ ]:
sc.pl.umap(adata, color = 'cell_type_simplified')

In [ ]:
res = 0.1
sc.tl.leiden(adata, resolution=res, key_added="leiden_" + str(res))

In [ ]:
sc.pl.umap(adata, color = 'leiden_0.1')#, groups = '6')

# subset neurons

In [ ]:
adata.X = adata.layers['counts']

In [ ]:
np.unique(adata.layers["counts"].data)

In [ ]:
neurons = adata[adata.obs['cell_type_simplified']=='neurons']

In [ ]:
adata.obs['cell_type_simplified'].value_counts()

In [ ]:
neurons.obs['cell_type_simplified'].value_counts()

In [ ]:
sc.pp.normalize_total(neurons)
sc.pp.log1p(neurons)
neurons.layers["log1p_norm"] = neurons.X.copy()
sc.pp.highly_variable_genes(neurons,n_top_genes=3000, subset=False)
sc.tl.pca(neurons)
sc.pp.neighbors(neurons)
sc.tl.umap(neurons)
neurons

In [ ]:
res = 0.5
sc.tl.leiden(neurons, resolution=res, key_added="leiden_" + str(res))

In [ ]:
sc.pl.umap(neurons, color = "leiden_0.5")

In [ ]:
marker_genes = {
    'neurons' : ['INA','SYT1','DCX'],
    'GlutaN' : ['SLC17A6', 'SLC17A7'],
    'GabaN':['SLC32A1','GAD1','GAD2'],
    'DopaN':['TH','SLC6A3','SLC18A2'],
    'Serotogenic':['FEV','SLC6A4','TPH2','CHGB']
}

In [ ]:
marker_genes_in_data = dict()
for ct, markers in marker_genes.items():
    markers_found = list()
    for marker in markers:
        if marker in adata.var.index:
            markers_found.append(marker)
    marker_genes_in_data[ct] = markers_found

In [ ]:
sc.pl.dotplot(
    neurons,
    groupby="leiden_0.5",
    var_names=marker_genes_in_data,
    standard_scale="var", 
)

In [ ]:
nclusters = len(neurons.obs['leiden_0.5'].unique())
for cluster in range(0,nclusters):
    print(f'"{cluster}":"{cluster}__Unk",')

In [ ]:
res = 0.5
cluster2annotation = {
    "0":"0__GlutaN",
"1":"1__GabaN",
"2":"2__GabaN",
"3":"3__neurons_unk",
"4":"4__DopaN",
"5":"5__GlutaN",
"6":"6__GlutaN",
"7":"7__DopaN",
"8":"8__GabaN",
"9":"9__GlutaN",
"10":"10__GlutaN",
"11":"11__GabaN",
"12":"12__GabaN",
"13":"13__neurons_unk",
"14":"14__GlutaN",
"15":"15__GlutaN",
"16":"16__GabaN",
"17":"17__GabaN",
"18":"18__neurons_unk",
"19":"19__GabaN",
"20":"20__GabaN",
"21":"21__GabaN",
"22":"22__GabaN",
"23":"23__GlutaN",
"24":"24__GlutaN",
"25":"25__GlutaN",
"26":"26__GlutaN",
"27":"27__GlutaN",
"28":"28__neurons_unk",
"29":"29__GlutaN",
"30":"30__DopaN",
"31":"31__neurons_unk",
"32":"32__GabaN",
"33":"33__DopaN",
"34":"34__neurons_unk",
}

In [ ]:
neurons.obs["cell_type_neurons_" + str(res)] = (
    neurons.obs["leiden_" + str(res)].map(cluster2annotation).astype("category")
)

In [ ]:
neurons.obs["cell_type_neurons_"  + str(res) + "_simplified"] = [x.split('__')[1] for x in neurons.obs["cell_type_neurons_" + str(res)]]

In [ ]:
sc.pl.umap(neurons, color = "cell_type_neurons_0.5_simplified")

In [ ]:
neurons.obs['cell_type_neurons_0.5_simplified'].value_counts()

In [ ]:
sc.pl.umap(neurons,color = ['TH','SLC6A3','SLC18A2'])

In [ ]:
sc.pl.umap(neurons,color = ['MALAT1'])

In [ ]:
neurons.obs 

In [ ]:
adata

In [ ]:
date = datetime.datetime.now().strftime('%Y%m%d')
neurons.obs['cell_type_neurons_0.5_simplified'].to_csv(f'miscellaneous/{date}_neurons_annot_sn.csv')

In [ ]:
sc.pl.umap(neurons,color = ['vireo_assignment'])

In [ ]:
neurons.obs['vireo_assignment'].value_counts()

In [ ]:
neurons.obs['vireo_assignment'][neurons.obs['cell_type_neurons_0.5_simplified']=='DopaN'].value_counts()[:50]

# Map neuronal annotation back to adata

In [ ]:
adata.obs

In [ ]:
neurons.obs['cell_type_neurons_0.5_simplified'].value_counts()

In [ ]:
# map annotaition
index_value_dict = neurons.obs['cell_type_neurons_0.5_simplified'].to_dict()

adata.obs['cell_type_neuron_incl'] = adata.obs['cell_type_simplified']

adata.obs['cell_type_neuron_incl'] = adata.obs.apply(
    lambda row: index_value_dict.get(row.name, row['cell_type_neuron_incl']),
    axis=1
)

In [ ]:
sc.pl.umap(adata, color = 'cell_type_neuron_incl')

In [ ]:
adata.obs['cell_type_neuron_incl'].value_counts()

In [ ]:
adata.obs['cell_type_simplified'].value_counts()

# Add immune cells

In [ ]:
# Add immune
sc.pl.umap(adata, color = 'leiden_0.1', groups = '8')


In [ ]:
adata.X = adata.layers['log1p_norm']
sc.pl.umap(adata, color = ['GNLY','RUNX1',"MS4A1","CD8A","IL7R"])

In [ ]:
def check_columns(row):
    if (row['leiden_0.1']=='8'):
        return 'immune'
    else:
        return row['cell_type_neuron_incl']


adata.obs['cell_type_neuron_incl'] = adata.obs.apply(lambda row: check_columns(row), axis=1)

In [ ]:
adata.obs['cell_type_neuron_incl'] = adata.obs['cell_type_neuron_incl'].apply(lambda x: 'Astro' if x == 'Ependyma' else x)


In [ ]:
sc.pl.umap(adata, color = 'cell_type_neuron_incl')

In [ ]:
adata.obs

In [ ]:
date = datetime.datetime.now().strftime('%Y%m%d')

adata.write(f'./hdf5/{date}_all_data_integrated_filtered_umap_scANVI_metadata_ct_corrected.sn.h5ad')